#### Adds:
* Precision@K
* Recall@K
* Retrieval Accuracy
* Benchmark Dataset
* Performance Analysis
* Evaluation Framework

### Imports

In [1]:
import pandas as pd

### Create Evaluation Dataset
These are our ground-truth questions.

In [2]:
evaluation_data = [

    {
        "question": "How many leave days are allowed?",
        "expected_source": "employee_handbook.txt"
    },

    {
        "question": "What monitoring tools are used?",
        "expected_source": "cloud_migration_guide.txt"
    },

    {
        "question": "Who works in ML Engineering?",
        "expected_source": "employees.csv"
    },

    {
        "question": "Which cloud platform is used?",
        "expected_source": "cloud_migration_guide.txt"
    },

    {
        "question": "How often should passwords be changed?",
        "expected_source": "company_policies.txt"
    }

]

print("Evaluation dataset created!")

Evaluation dataset created!


### Verify Dataset

In [3]:
pd.DataFrame(evaluation_data)

,question,expected_source
0,How many leave days are allowed?,employee_handbook.txt
1,What monitoring tools are used?,cloud_migration_guide.txt
2,Who works in ML Engineering?,employees.csv
3,Which cloud platform is used?,cloud_migration_guide.txt
4,How often should passwords be changed?,company_policies.txt


### Load Required Files

In [4]:
import pickle
import faiss

### Load metadata:

In [5]:
with open("data/metadata.pkl", "rb") as f:
    metadata = pickle.load(f)

### Load vectorizer:

In [6]:
with open("data/vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)

### Load FAISS index:

In [7]:
index = faiss.read_index("data/faiss_index.bin")

print("Files loaded successfully!")

Files loaded successfully!


###  Recreate Hybrid Search

In [8]:
def hybrid_search(question, top_k=3):

    query_vector = vectorizer.transform([question]).toarray().astype("float32")

    distances, indices = index.search(query_vector, top_k)

    results = []

    for idx in indices[0]:

        results.append({

            "source": metadata[idx]["source"],

            "text": metadata[idx]["text"]

        })

    return results

### Precision@K

In [9]:
def precision_at_k(retrieved_sources, expected_source):

    relevant = 0

    for source in retrieved_sources:

        if source == expected_source:

            relevant += 1

    return relevant / len(retrieved_sources)

### Recall@K
Since each query has one correct source:

In [10]:
def recall_at_k(retrieved_sources, expected_source):

    if expected_source in retrieved_sources:

        return 1

    return 0

### Evaluate Entire System

In [11]:
results = []

for item in evaluation_data:

    retrieved = hybrid_search(item["question"])

    sources = [x["source"] for x in retrieved]

    precision = precision_at_k(

        sources,

        item["expected_source"]

    )

    recall = recall_at_k(

        sources,

        item["expected_source"]

    )

    results.append({

        "question": item["question"],

        "expected_source": item["expected_source"],

        "retrieved_sources": sources,

        "precision@3": precision,

        "recall@3": recall

    })

print("Evaluation completed!")

Evaluation completed!


####  Results Table

In [12]:
evaluation_df = pd.DataFrame(results)

evaluation_df

,question,expected_source,retrieved_sources,precision@3,recall@3
0,How many leave days are allowed?,employee_handbook.txt,"[employee_handbook.txt, company_policies.txt, ...",0.333333,1
1,What monitoring tools are used?,cloud_migration_guide.txt,"[cloud_migration_guide.txt, employee_handbook....",0.333333,1
2,Who works in ML Engineering?,employees.csv,"[employees.csv, support_tickets.csv, cloud_mig...",0.333333,1
3,Which cloud platform is used?,cloud_migration_guide.txt,"[cloud_migration_guide.txt, company_policies.t...",0.333333,1
4,How often should passwords be changed?,company_policies.txt,"[company_policies.txt, cloud_migration_guide.t...",0.333333,1


###  Average Precision

In [13]:
avg_precision = evaluation_df["precision@3"].mean()

print("Average Precision@3 =", round(avg_precision, 3))

Average Precision@3 = 0.333


### Average Recall

In [18]:
avg_recall = evaluation_df["recall@3"].mean()

print("Average Recall@3 =", round(avg_recall, 3))

Average Recall@3 = 1.0


### Retrieval Accuracy
Accuracy = expected source appears anywhere in Top-K.

In [19]:
accuracy = (

    evaluation_df["recall@3"]

    .mean()

    * 100

)

print("Retrieval Accuracy =", round(accuracy, 2), "%")

Retrieval Accuracy = 100.0 %


### Save Results

In [20]:
evaluation_df.to_csv(

    "outputs/evaluation_metrics.csv",

    index=False

)

print("Evaluation report exported!")

Evaluation report exported!


### Verify Outputs

In [21]:
import os

print(os.listdir("outputs"))

['evaluation_metrics.csv', 'query_analytics.csv']


### Final Metrics Summary

In [22]:
print("=" * 50)

print("Precision@3 :", round(avg_precision, 3))

print("Recall@3 :", round(avg_recall, 3))

print("Retrieval Accuracy :", round(accuracy, 2), "%")

print("=" * 50)

Precision@3 : 0.333
Recall@3 : 1.0
Retrieval Accuracy : 100.0 %
